In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_parquet('../data/processed/portfolio_with_risk_params.parquet')

# Ambil portofolio aktif (Stage 1, 2, dan 3) yang membutuhkan pencadangan CKPN
active_df = df[df['psak71_stage'] > 0].copy()
print(f"Total rekening aktif dalam portofolio: {len(active_df):,}")

# Matriks Transisi Tahunan (Annual Transition Matrix) berbasis empiris perbankan
# Baris/Kolom: [Stage 1 (Lancar), Stage 2 (DPK/SICR), Stage 3 (Default/Absorbing), Lunas (Prepaid/Absorbing)]
# Catatan: Stage 3 dan Lunas bersifat absorbing state (1.0 pada diagonal) untuk proyeksi kerugian kumulatif
transition_matrix = np.array([
    [0.82, 0.08, 0.03, 0.07],  # Dari Stage 1 -> 82% tetap S1, 8% turun ke S2, 3% Default, 7% Lunas
    [0.15, 0.55, 0.25, 0.05],  # Dari Stage 2 -> 15% pulih ke S1, 55% tetap S2, 25% Default, 5% Lunas
    [0.00, 0.00, 1.00, 0.00],  # Dari Stage 3 -> Absorbing Default (100%)
    [0.00, 0.00, 0.00, 1.00]   # Dari Lunas   -> Absorbing Lunas (100%)
])

# Hitung matriks transisi multi-tahun P^t (Tahun ke-1 hingga Tahun ke-5) menggunakan aljabar matriks
P_1 = transition_matrix
P_2 = np.linalg.matrix_power(transition_matrix, 2)
P_3 = np.linalg.matrix_power(transition_matrix, 3)
P_5 = np.linalg.matrix_power(transition_matrix, 5)

print("\n=== Cumulative Default Probability (Kolom ke-3) dari Markov Chain ===")
print(f"Stage 1 -> Default (1Y / 3Y / 5Y): {P_1[0,2]:.2%} / {P_3[0,2]:.2%} / {P_5[0,2]:.2%}")
print(f"Stage 2 -> Default (1Y / 3Y / 5Y): {P_1[1,2]:.2%} / {P_3[1,2]:.2%} / {P_5[1,2]:.2%}")

Total rekening aktif dalam portofolio: 1,089,961

=== Cumulative Default Probability (Kolom ke-3) dari Markov Chain ===
Stage 1 -> Default (1Y / 3Y / 5Y): 3.00% / 12.25% / 21.07%
Stage 2 -> Default (1Y / 3Y / 5Y): 25.00% / 47.68% / 56.71%


In [2]:
# Hitung faktor pengali (scalar multiplier) dari Markov Chain untuk tenor 3 tahun dan 5 tahun
mult_s1_3y = P_3[0, 2] / P_1[0, 2]
mult_s1_5y = P_5[0, 2] / P_1[0, 2]
mult_s2_3y = P_3[1, 2] / P_1[1, 2]
mult_s2_5y = P_5[1, 2] / P_1[1, 2]

def compute_lifetime_pd(row):
    if row['psak71_stage'] == 3:
        return 1.0
    elif row['psak71_stage'] == 1:
        mult = mult_s1_3y if row['tenor_months'] == 36 else mult_s1_5y
        return min(row['pd_12m'] * mult, 0.99)
    else: # Stage 2
        mult = mult_s2_3y if row['tenor_months'] == 36 else mult_s2_5y
        return min(row['pd_12m'] * mult, 0.99)

active_df['pd_lifetime'] = active_df.apply(compute_lifetime_pd, axis=1)

# Tentukan Effective PD sesuai aturan PSAK 71:
# Stage 1 menggunakan 12-Month PD; Stage 2 & Stage 3 menggunakan Lifetime PD
active_df['effective_pd_base'] = np.where(
    active_df['psak71_stage'] == 1,
    active_df['pd_12m'],
    active_df['pd_lifetime']
)

print("Rata-rata Effective PD (Baseline) per Stage:")
print(active_df.groupby('psak71_stage')['effective_pd_base'].mean().round(4))

Rata-rata Effective PD (Baseline) per Stage:
psak71_stage
1    0.0588
2    0.2180
3    1.0000
Name: effective_pd_base, dtype: float64


In [3]:
# 1. Hitung tingkat default historis bulanan berdasarkan tanggal pencairan (Cohort Default Rate)
cohort_df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off', 'Default'])].groupby('issue_date').agg(
    default_rate=('is_default', 'mean'),
    loan_count=('id', 'count')
).reset_index()

# 2. Simulasi data makroekonomi historis (Suku Bunga Acuan & Pengangguran) sejajar periode 2015-2018
#    (Anda juga dapat menggantinya dengan file CSV makro riil dari FRED / Bank Indonesia)
np.random.seed(42)
n_months = len(cohort_df)
cohort_df['interest_rate_macro'] = np.linspace(0.25, 2.25, n_months) + np.random.normal(0, 0.05, n_months)
cohort_df['unemployment_macro'] = np.linspace(5.5, 3.8, n_months) + np.random.normal(0, 0.1, n_months)

# 3. Estimasi sensitivitas PD terhadap perubahan Suku Bunga & Pengangguran menggunakan Regresi Linear
from sklearn.linear_model import LinearRegression

macro_model = LinearRegression()
macro_model.fit(cohort_df[['interest_rate_macro', 'unemployment_macro']], cohort_df['default_rate'])
beta_rate, beta_unemp = macro_model.coef_

print(f"Sensitivitas PD terhadap Kenaikan Suku Bunga (+1%) : {beta_rate:+.4f}")
print(f"Sensitivitas PD terhadap Kenaikan Pengangguran (+1%): {beta_unemp:+.4f}")

# 4. Definisikan 3 Skenario Forward-Looking (Perubahan variabel makro 12 bulan ke depan)
#    - Optimistic (Bobot 25%): Suku bunga turun 0.5%, pengangguran turun 0.5%
#    - Base Case  (Bobot 50%): Suku bunga naik 0.25%, pengangguran stabil 0.0%
#    - Downturn   (Bobot 25%): Suku bunga naik 2.0%, pengangguran naik 1.5% (Simulasi Krisis)

delta_pd_opt  = (beta_rate * -0.50) + (beta_unemp * -0.50)
delta_pd_base = (beta_rate * +0.25) + (beta_unemp *  0.00)
delta_pd_down = (beta_rate * +2.00) + (beta_unemp * +1.50)

# Terapkan penyesuaian makro pada PD (kecuali Stage 3 yang sudah 100% default)
for scen, delta in [('opt', delta_pd_opt), ('base', delta_pd_base), ('down', delta_pd_down)]:
    active_df[f'pd_{scen}'] = np.where(
        active_df['psak71_stage'] == 3,
        1.0,
        (active_df['effective_pd_base'] + max(delta, -0.01)).clip(0.001, 0.99)
    )

# 5. Probability-Weighted PD (Sesuai standar PSAK 71: 25% Opt + 50% Base + 25% Downturn)
active_df['pd_weighted'] = (
    0.25 * active_df['pd_opt'] +
    0.50 * active_df['pd_base'] +
    0.25 * active_df['pd_down']
)

Sensitivitas PD terhadap Kenaikan Suku Bunga (+1%) : +0.0067
Sensitivitas PD terhadap Kenaikan Pengangguran (+1%): +0.0691


In [4]:
# Hitung nominal ECL untuk tiap skenario (ECL = PD * LGD * EAD)
active_df['ecl_optimistic'] = active_df['pd_opt'] * active_df['segment_lgd'] * active_df['ead']
active_df['ecl_base']       = active_df['pd_base'] * active_df['segment_lgd'] * active_df['ead']
active_df['ecl_downturn']   = active_df['pd_down'] * active_df['segment_lgd'] * active_df['ead']
active_df['ecl_weighted']   = active_df['pd_weighted'] * active_df['segment_lgd'] * active_df['ead']

# Buat tabel ringkasan CKPN per Stage (Laporan Eksekutif Manajemen Risiko)
ckpn_report = active_df.groupby('psak71_stage').agg(
    Jumlah_Rekening=('id', 'count'),
    Total_EAD=('ead', 'sum'),
    Rata2_PD_Weighted=('pd_weighted', 'mean'),
    Rata2_LGD=('segment_lgd', 'mean'),
    ECL_Base=('ecl_base', 'sum'),
    ECL_Downturn=('ecl_downturn', 'sum'),
    ECL_Weighted_PSAK71=('ecl_weighted', 'sum')
).reset_index()

ckpn_report['Coverage_Ratio_%'] = (ckpn_report['ECL_Weighted_PSAK71'] / ckpn_report['Total_EAD']) * 100

print("=== LAPORAN PERHITUNGAN CKPN / ECL BERBASIS PSAK 71 ===")
print(ckpn_report.to_string(index=False))

print(f"\nTotal Eksposur Kredit Aktif (EAD) : ${active_df['ead'].sum():,.2f}")
print(f"Total Kebutuhan Cadangan (CKPN)   : ${active_df['ecl_weighted'].sum():,.2f}")
print(f"Total CKPN Skenario Krisis        : ${active_df['ecl_downturn'].sum():,.2f}")

# Simpan dataset final yang sudah lengkap untuk Dashboard Streamlit
active_df.to_parquet('../data/processed/final_ecl_portfolio.parquet', index=False)

=== LAPORAN PERHITUNGAN CKPN / ECL BERBASIS PSAK 71 ===
 psak71_stage  Jumlah_Rekening    Total_EAD  Rata2_PD_Weighted  Rata2_LGD     ECL_Base  ECL_Downturn  ECL_Weighted_PSAK71  Coverage_Ratio_%
            1           866142 9.059002e+09           0.086484   0.943450 4.836309e+08  1.470682e+09         7.055598e+08          7.788494
            2            31863 3.867362e+08           0.245683   0.934601 7.935566e+07  1.210658e+08         8.872839e+07         22.942873
            3           191956 3.010535e+09           1.000000   0.938811 2.824251e+09  2.824251e+09         2.824251e+09         93.812247

Total Eksposur Kredit Aktif (EAD) : $12,456,273,383.64
Total Kebutuhan Cadangan (CKPN)   : $3,618,538,896.60
Total CKPN Skenario Krisis        : $4,415,998,962.48


In [5]:
import pandas as pd
import os

# Baca file final yang berukuran 90 MB tadi
df_full = pd.read_parquet('../data/processed/final_ecl_portfolio.parquet')

# Pilih 12 kolom yang dipakai dashboard saja (100% baris tetap utuh)
cols_dashboard = [
    'id', 'grade', 'sub_grade', 'psak71_stage', 'tenor_months',
    'ead', 'segment_lgd', 'pd_weighted', 'ecl_optimistic',
    'ecl_base', 'ecl_downturn', 'ecl_weighted'
]

df_optimized = df_full[cols_dashboard].copy()

# Optimasi tipe data (float64 -> float32) tanpa mengurangi presisi angka
float_cols = ['ead', 'segment_lgd', 'pd_weighted', 'ecl_optimistic', 'ecl_base', 'ecl_downturn', 'ecl_weighted']
df_optimized[float_cols] = df_optimized[float_cols].astype('float32')
df_optimized['psak71_stage'] = df_optimized['psak71_stage'].astype('int8')
df_optimized['tenor_months'] = df_optimized['tenor_months'].astype('int16')

# Timpa file lama dengan versi yang sudah terkompresi
output_path = '../data/processed/final_ecl_portfolio.parquet'
df_optimized.to_parquet(output_path, index=False, compression='snappy')

size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"Berhasil! Total baris: {len(df_optimized):,} (100% UTUH) | Ukuran baru: {size_mb:.2f} MB")

Berhasil! Total baris: 1,089,961 (100% UTUH) | Ukuran baru: 32.62 MB
